# Backtesting VaR

In [1]:
import os
from typing import Tuple
import numpy as np
import pandas as pd
import plotly.express as px
pd.options.plotting.backend = "plotly"

import sys
sys.path.append(os.getcwd().split("scripts")[0])
sys.path.append(os.path.join(os.getcwd().split("scripts")[0], "params"))

from tqdm import tqdm
from params import funding, caps
import pystable

In [2]:
filename = "btc"

path_to_file = os.path.join(os.getcwd().split("scripts")[0], f"data/{filename}")

periodicity = 24 * 60 * 60. # 1 day in seconds

In [3]:
df = pd.read_csv(path_to_file+".csv", parse_dates=["timestamp"]).set_index("timestamp")
df

,close
timestamp,
2024-01-01 00:01:00+00:00,42298.61
2024-01-01 00:02:00+00:00,42320.00
2024-01-01 00:03:00+00:00,42325.50
2024-01-01 00:04:00+00:00,42367.99
2024-01-01 00:05:00+00:00,42397.23
...,...
2024-05-08 13:06:00+00:00,62097.67
2024-05-08 13:07:00+00:00,62077.49
2024-05-08 13:08:00+00:00,62115.51


In [4]:
def compute_log_return(_df:pd.DataFrame, column:str) -> pd.DataFrame:
    tmp_df = _df.copy()
    if not column in tmp_df.columns:
        raise Exception(f"Column {column} not found")
    tmp_df["log_return"] = np.nan
    tmp_df.loc[tmp_df.index[1]:,"log_return"] = np.log(
        tmp_df[column].iloc[1:].to_numpy() / tmp_df[column].iloc[:-1].to_numpy()
    )
    tmp_df.dropna(inplace=True)
    return tmp_df

In [5]:
df = compute_log_return(df, "close")
df

,close,log_return
timestamp,,
2024-01-01 00:02:00+00:00,42320.00,0.000506
2024-01-01 00:03:00+00:00,42325.50,0.000130
2024-01-01 00:04:00+00:00,42367.99,0.001003
2024-01-01 00:05:00+00:00,42397.23,0.000690
2024-01-01 00:06:00+00:00,42409.20,0.000282
...,...,...
2024-05-08 13:06:00+00:00,62097.67,-0.000543
2024-05-08 13:07:00+00:00,62077.49,-0.000325
2024-05-08 13:08:00+00:00,62115.51,0.000612


In [6]:
df_lowest = df[["close"]].resample("1D", label="right").min()
df_lowest = compute_log_return(df_lowest, "close")
df_lowest

,close,log_return
timestamp,,
2024-01-03 00:00:00+00:00,44164.97,0.045357
2024-01-04 00:00:00+00:00,40887.99,-0.077096
2024-01-05 00:00:00+00:00,42631.99,0.041769
2024-01-06 00:00:00+00:00,42529.75,-0.002401
2024-01-07 00:00:00+00:00,43430.01,0.020947
...,...,...
2024-05-05 00:00:00+00:00,62592.27,0.062101
2024-05-06 00:00:00+00:00,62927.99,0.005349
2024-05-07 00:00:00+00:00,62795.54,-0.002107


In [7]:
df_daily = df[["close"]].resample("1D", label="right").last()
df_daily

,close
timestamp,
2024-01-02 00:00:00+00:00,44156.10
2024-01-03 00:00:00+00:00,44933.03
2024-01-04 00:00:00+00:00,42829.49
2024-01-05 00:00:00+00:00,44133.07
2024-01-06 00:00:00+00:00,44118.07
...,...
2024-05-05 00:00:00+00:00,63878.03
2024-05-06 00:00:00+00:00,63991.12
2024-05-07 00:00:00+00:00,63165.20


In [8]:
df_daily = compute_log_return(df_daily, "close")
df_daily

,close,log_return
timestamp,,
2024-01-03 00:00:00+00:00,44933.03,0.017442
2024-01-04 00:00:00+00:00,42829.49,-0.047946
2024-01-05 00:00:00+00:00,44133.07,0.029983
2024-01-06 00:00:00+00:00,44118.07,-0.000340
2024-01-07 00:00:00+00:00,43962.28,-0.003537
...,...,...
2024-05-05 00:00:00+00:00,63878.03,0.016165
2024-05-06 00:00:00+00:00,63991.12,0.001769
2024-05-07 00:00:00+00:00,63165.20,-0.012991


In [9]:
def fit_distribution(
    _df:pd.DataFrame, column:str="log_return", period:float=periodicity, verbose:bool=False
) -> Tuple[pystable.STABLE_DIST, pystable.STABLE_DIST]:

    if not column in _df.columns:
        raise Exception(f"Column {column} not found")

    _orig_dst = funding.gaussian()
    pystable.fit(_orig_dst, _df[column].to_numpy(), _df.index.size)

    if verbose:
        print(f'''
            original fitted distribution
            alpha: {_orig_dst.contents.alpha}, beta: {_orig_dst.contents.beta},
            mu: {_orig_dst.contents.mu_1}, sigma: {_orig_dst.contents.sigma}
            '''
        )

    # this rescale changes from the original unit of the data to seconds
    _rescaled_dst = caps.rescale(_orig_dst, 1./period)
    _rescaled_dst_2 = funding.rescale(_orig_dst, 1./period)

    if verbose:
        print(f'''
            (caps.py) rescaled distribution (1/t = {1./period}):
            alpha: {_rescaled_dst.contents.alpha}, beta: {_rescaled_dst.contents.beta},
            mu: {_rescaled_dst.contents.mu_1}, sigma: {_rescaled_dst.contents.sigma}
            '''
        )
        print(f'''
            (funding.py) rescaled distribution (1/t = {1./period}):
            alpha: {_rescaled_dst_2.contents.alpha}, beta: {_rescaled_dst_2.contents.beta},
            mu: {_rescaled_dst_2.contents.mu_1}, sigma: {_rescaled_dst_2.contents.sigma}
            '''
        )

    return _orig_dst, _rescaled_dst

## Compute VaR coverage

In [10]:
orig_dst, rescaled_dst = fit_distribution(df_daily, column="log_return", period=periodicity, verbose=True)


            original fitted distribution
            alpha: 1.3649897916940414, beta: -0.10882300242143826,
            mu: -0.00016567415218295438, sigma: 0.014170593535141314
            

            (caps.py) rescaled distribution (1/t = 1.1574074074074073e-05):
            alpha: 1.3649897916940414, beta: -0.10882300242143826,
            mu: -1.917524909524935e-09, sigma: 4.303918616341934e-06
            

            (funding.py) rescaled distribution (1/t = 1.1574074074074073e-05):
            alpha: 1.3649897916940414, beta: -0.10882300242143826,
            mu: -1.917524909524935e-09, sigma: 3.426631558005301e-06
            


In [11]:
def VaR(_any_dst:pystable.STABLE_DIST, _forecast:int, _alphas: np.ndarray) -> np.ndarray:
    """
    Compute F^{-1}_{X_t}(1-alpha) using pystable
    * _any_dst: any distribution
    * _forecast: the forecast period (in the unit of the distribution)
    * _alphas: the 
    """
    _dst_y = _any_dst

    # this rescale changes from seconds to "_forecast"
    if np.abs(_forecast - 1.) > 1e-15:
        _dst_y = caps.rescale(_any_dst, _forecast)

    return np.array(pystable.q(_dst_y, _alphas, len(_alphas)))

In [18]:
one_day = VaR(orig_dst, 1, funding.ALPHAS)
one_day

array([-0.15890017, -0.0831583 , -0.05138857, -0.03862454, -0.03123145])

In [19]:
df_lowest.loc[df_lowest.index[-1], "log_return"]

-0.004155721594036579

In [21]:
one_day_scaled = VaR(rescaled_dst, periodicity, funding.ALPHAS)
one_day_scaled

array([-0.15890017, -0.0831583 , -0.05138857, -0.03862454, -0.03123145])